# **Collegamento Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/7ImmAugmentation/data/input_images/
!ls /content/drive/MyDrive/7ImmAugmentation/results/weights/
!ls /content/drive/MyDrive/7ImmAugmentation/results/losses/
!ls /content/drive/MyDrive/7ImmAugmentation/results/sample/

Mounted at /content/drive
'Copia di Copia di 6.jpg'   'Copia di Copia di o.jpg'
'Copia di Copia di f.jpg'   'Copia di Copia di t.png'
'Copia di Copia di i.jpg'   'Copia di Copia di u.jpg'
'Copia di Copia di k.jpeg'


# **Setup**

In [ ]:
!pip install torchmetrics
!pip install lpips
!pip install pytorch-fid

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 962.6/962.6 kB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 122.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 94.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 112.3 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninsta

# **Metriche intraset**

In [ ]:
import os
import torch
from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure
import lpips
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt
from datetime import datetime

#from pytorch_fid import fid_score

def save_intraset_metrics_plot(metrics_history, dataset_dim):
    """
    Salva un grafico riassuntivo delle metriche PSNR, SSIM, LPIPS, FID durante il training.
    metrics_history è una lista di dict, es:
    [{'step': 0, 'psnr': ..., 'ssim': ..., 'lpips': ..., 'fid': ...}, ...]
    """
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

    steps = [m['step'] for m in metrics_history]
    psnr = [m['psnr'] for m in metrics_history]
    ssim = [m['ssim'] for m in metrics_history]
    lpips = [m['lpips'] for m in metrics_history]
    fid = [m['fid'] for m in metrics_history]

    fig, axs = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"Intraset Metrics Summary ({timestamp})", fontsize=18)

    # PSNR
    axs[0, 0].plot(steps, psnr, color='blue')
    axs[0, 0].set_title('PSNR')
    axs[0, 0].set_xlabel('Step')
    axs[0, 0].set_ylabel('PSNR (dB)')
    axs[0, 0].grid(True)

    # SSIM
    axs[0, 1].plot(steps, ssim, color='green')
    axs[0, 1].set_title('SSIM')
    axs[0, 1].set_xlabel('Step')
    axs[0, 1].set_ylabel('SSIM')
    axs[0, 1].grid(True)

    # LPIPS
    axs[1, 0].plot(steps, lpips, color='red')
    axs[1, 0].set_title('LPIPS')
    axs[1, 0].set_xlabel('Step')
    axs[1, 0].set_ylabel('LPIPS (lower is better)')
    axs[1, 0].grid(True)

    # FID
    axs[1, 1].plot(steps, fid, color='purple')
    axs[1, 1].set_title('FID')
    axs[1, 1].set_xlabel('Step')
    axs[1, 1].set_ylabel('FID (lower is better)')
    axs[1, 1].grid(True)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    save_dir='/content/drive/MyDrive/7ImmAugmentation/results/intraset'
    ensure_dir_exists(save_dir)
    save_path = os.path.join(save_dir, f"intraset_metrics_dataset_dim_{dataset_dim}.png")
    plt.savefig(save_path)
    plt.close(fig)
    print(f"Salvato Intraset metrics plot in: {save_path}")

def denormalize(tensor):
    return (tensor + 1) / 2  # [-1, 1] -> [0, 1]

def evaluate_intraset(real_batch, generated_batch, device='cuda'):
    psnr = PeakSignalNoiseRatio().to(device)
    ssim = StructuralSimilarityIndexMeasure().to(device)
    loss_fn = lpips.LPIPS(net='vgg').to(device)

    psnr_scores, ssim_scores, lpips_scores = [], [], []

    with torch.no_grad():  # 🔒 evita calcolo del grafo
        for real_img, gen_img in zip(real_batch, generated_batch):
            # Denormalizza da [-1, 1] a [0, 1]
            real_img = denormalize(real_img).unsqueeze(0).to(device)
            gen_img = denormalize(gen_img).unsqueeze(0).to(device)

            psnr_scores.append(psnr(gen_img, real_img).item())
            ssim_scores.append(ssim(gen_img, real_img).item())
            lpips_scores.append(loss_fn(gen_img, real_img).item())

    avg_psnr = sum(psnr_scores) / len(psnr_scores)
    avg_ssim = sum(ssim_scores) / len(ssim_scores)
    avg_lpips = sum(lpips_scores) / len(lpips_scores)

    return {
        'psnr': avg_psnr,
        'ssim': avg_ssim,
        'lpips': avg_lpips,
        'fid': None  # oppure 'not_computed'
    }



# **Utils**

In [ ]:
from datetime import datetime
import torch
import gc

def count_tensors():
    count_gpu = 0
    count_cpu = 0
    for obj in gc.get_objects():
        try:
            if torch.is_tensor(obj):
                if obj.is_cuda:
                    count_gpu += 1
                else:
                    count_cpu += 1
        except Exception:
            pass
    return count_cpu, count_gpu


def count_cpu_tensors():
    return sum(1 for obj in gc.get_objects() if torch.is_tensor(obj) and obj.device.type == 'cpu')

In [ ]:
import numpy as np

def find_well_trained_step(gen_losses, disc_acc_real, disc_acc_fake, window=1000, gen_loss_thresh=0.01, acc_range=(45, 55)):
    """
    Trova lo step a partire dal quale la rete è considerata sufficientemente addestrata.
    - window: numero di step consecutivi da controllare
    - gen_loss_thresh: variazione massima media del generatore (percentuale)
    - acc_range: intervallo accettabile di accuratezza (%) per il discriminatore
    """

    for i in range(len(gen_losses) - window):
        gen_window = gen_losses[i:i + window]
        acc_real_window = disc_acc_real[i:i + window]
        acc_fake_window = disc_acc_fake[i:i + window]

        gen_var = np.abs((gen_window[-1] - gen_window[0]) / (gen_window[0] + 1e-8))
        acc_real_ok = np.all((np.array(acc_real_window) >= acc_range[0]) & (np.array(acc_real_window) <= acc_range[1]))
        acc_fake_ok = np.all((np.array(acc_fake_window) >= acc_range[0]) & (np.array(acc_fake_window) <= acc_range[1]))

        if gen_var < gen_loss_thresh and acc_real_ok and acc_fake_ok:
            return i + window  # restituisce lo step dove inizia la stabilità
    return None  # non trovato


In [ ]:
import torch.nn as nn

def weights_init_normal(m):
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
        if m.bias is not None:
            nn.init.constant_(m.bias.data, 0.0)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0.0)

In [ ]:
import os
import torch
import glob

def load_weights(generator, discriminator, weights_dir='/content/drive/MyDrive/7ImmAugmentation/results/weights'):
    # Assicurati che la cartella esista
    os.makedirs(weights_dir, exist_ok=True)

    # Cerca tutti i file che corrispondono al pattern
    weight_files = glob.glob(os.path.join(weights_dir, 'weights_*.pt'))

    if not weight_files:
        print("Nessun file di pesi trovato. Inizializzazione dei pesi.")
        return 0, {}, {}, {}  # Nessuno step, nessuna loss, nessuna acc

    # Ordina i file in base al timestamp nel nome (decrescente, più recente per primo)
    weight_files.sort(reverse=True)
    latest_weight_file = weight_files[0]

    # Carica il checkpoint solo per controllare 'step'
    checkpoint = torch.load(latest_weight_file, map_location='cpu')
    step = checkpoint.get('step', 0)

    if step == 50000:
        print(f"File {latest_weight_file} trovato, ma step == 50000. Inizializzazione dei pesi.")
        return 0, {}, {}

    # Carica i pesi nei modelli
    generator.load_state_dict(checkpoint['generator_state_dict'])
    discriminator.load_state_dict(checkpoint['discriminator_state_dict'])

    print(f"Pesi caricati da {latest_weight_file}, step {step}")

    # Recupera metriche se presenti
    losses = {
        'losses_D_list': checkpoint.get('losses_D_list', []),
        'losses_G_list': checkpoint.get('losses_G_list', []),
        'losses_G_adv_list': checkpoint.get('losses_G_adv_list', []),
        'losses_L1_list': checkpoint.get('losses_L1_list', []),
        'losses_style_list': checkpoint.get('losses_style_list', []),
        'losses_perceptual_list': checkpoint.get('losses_perceptual_list', [])
    }

    accuracies = {
        'acc_real_list': checkpoint.get('acc_real_list', []),
        'acc_fake_list': checkpoint.get('acc_fake_list', [])
    }

    metrics_history = checkpoint.get('metrics_history', [])

    return checkpoint.get('step', 0) + 1, losses, accuracies, metrics_history

In [ ]:
import torch
import matplotlib.pyplot as plt
import os
import torchvision.transforms.functional as TF

def denormalize(tensor):
    return (tensor + 1) / 2


def ensure_dir_exists(file_path: str) -> None:
    directory = os.path.dirname(file_path)
    if not os.path.exists(directory):
        os.makedirs(directory)


def moving_average(data, window_size):
    if len(data) < window_size:
        return []
    return [sum(data[i:i+window_size])/window_size for i in range(len(data) - window_size + 1)]


def visualize_sample(source, target, output, step=0, save_dir='/content/drive/MyDrive/7ImmAugmentation/results/sample'):
    source_detached = source.detach().cpu()
    target_detached = target.detach().cpu()
    output_detached = output.detach().cpu()

    source_img = denormalize(source_detached[0]).clamp(0, 1)
    target_img = denormalize(target_detached[0]).clamp(0, 1)
    output_img = denormalize(output_detached[0]).clamp(0, 1)

    fig, axs = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle(f"Training Step: {step}", fontsize=16)

    axs[0].imshow(TF.to_pil_image(source_img))
    axs[0].set_title("Input 128x128")

    axs[1].imshow(TF.to_pil_image(target_img))
    axs[1].set_title("Target 256x256")

    axs[2].imshow(TF.to_pil_image(output_img))
    axs[2].set_title("Generated 256x256")

    for ax in axs:
        ax.axis('off')

    plt.tight_layout()
    save_path = os.path.join(save_dir, f"sample_step{step:05}.png")
    ensure_dir_exists(save_path)
    plt.savefig(save_path)
    plt.close(fig)


def visualize_losses(losses_D, losses_G, timestamp, best_step=None):
    plt.figure(figsize=(10,5))
    plt.plot(losses_D, label='Loss Discriminatore')
    plt.plot(losses_G, label='Loss Generatore')
    if len(losses_D) >= 200:
        avg_D = moving_average(losses_D, 200)
        avg_G = moving_average(losses_G, 200)
        plt.plot(range(199, len(losses_D)), avg_D, label='Media 200 Loss D', linestyle='--')
        plt.plot(range(199, len(losses_G)), avg_G, label='Media 200 Loss G', linestyle='--')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title(f'Andamento delle loss (start: {timestamp})')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    save_dir='/content/drive/MyDrive/7ImmAugmentation/results/losses'
    ensure_dir_exists(save_dir)
    save_path = os.path.join(save_dir, f'loss_plot_{timestamp}.png')
    plt.savefig(save_path)
    plt.close()


def visualize_generator_loss_components(losses_G_adv, losses_L1, losses_style, losses_perceptual, timestamp):
    plt.figure(figsize=(10, 6))
    plt.plot(losses_G_adv, label='Adversarial Loss', color='blue')
    plt.plot(losses_L1, label='L1 Loss', color='green')
    plt.plot(losses_style, label='Style Loss', color='red')
    plt.plot(losses_perceptual, label='Perceptual Loss', color='purple')
    if len(losses_G_adv) >= 200:
        plt.plot(range(199, len(losses_G_adv)), moving_average(losses_G_adv, 200), linestyle='--', label='Media 200 Adv', color='blue')
        plt.plot(range(199, len(losses_L1)), moving_average(losses_L1, 200), linestyle='--', label='Media 200 L1', color='green')
        plt.plot(range(199, len(losses_style)), moving_average(losses_style, 200), linestyle='--', label='Media 200 Style', color='red')
        plt.plot(range(199, len(losses_perceptual)), moving_average(losses_perceptual, 200), linestyle='--', label='Media 200 Percep', color='purple')
    plt.xlabel('Step')
    plt.ylabel('Loss')
    plt.title(f'Componenti della Loss del Generatore (start: {timestamp})')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    save_dir='/content/drive/MyDrive/7ImmAugmentation/results/losses'
    ensure_dir_exists(save_dir)
    save_path = os.path.join(save_dir, f'generator_componnents_plot_{timestamp}.png')
    plt.savefig(save_path)
    plt.close()


def visualize_discriminator_accuracy(acc_real_list, acc_fake_list, timestamp):
    plt.figure(figsize=(10, 5))
    plt.plot(acc_real_list, label='Accuratezza su reali', color='green')
    plt.plot(acc_fake_list, label='Accuratezza su fake', color='red')
    if len(acc_real_list) >= 200:
        plt.plot(range(199, len(acc_real_list)), moving_average(acc_real_list, 200), linestyle='--', label='Media 200 Real', color='green')
        plt.plot(range(199, len(acc_fake_list)), moving_average(acc_fake_list, 200), linestyle='--', label='Media 200 Fake', color='red')
    plt.xlabel('Step')
    plt.ylabel('Accuratezza (%)')
    plt.title(f'Andamento accuratezza Discriminatore (start: {timestamp})')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    save_dir='/content/drive/MyDrive/7ImmAugmentation/results/losses'
    ensure_dir_exists(save_dir)
    save_path = os.path.join(save_dir, f"discriminator_accuracy_plot_{timestamp}.png")
    plt.savefig(save_path)
    plt.close()



# **Generatore**

In [ ]:
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(channels)
        )

    def forward(self, x):
        return x + self.block(x)

class Generator(nn.Module):
    def __init__(self):
        super().__init__()

        # Encoder: 3 convoluzioni iniziali (2 con stride 2)
        self.encoder = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 256, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
        )

        # Residual blocks
        self.res_blocks = nn.Sequential(*[ResidualBlock(256) for _ in range(6)])

        # Aumentiamo i canali a 512
        self.up_channel = nn.Sequential(
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )

        # Decoder: 3 upsampling (deconv strided)
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(512, 256, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(256, 128, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.ConvTranspose2d(128, 64, kernel_size=3, stride=2, padding=1, output_padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        # Output layer: 3 canali, Tanh
        self.final = nn.Sequential(
            nn.Conv2d(64, 3, kernel_size=7, padding=3),
            nn.Tanh()
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.res_blocks(x)
        x = self.up_channel(x)
        x = self.decoder(x)
        x = self.final(x)
        return x



# **Discriminatore**

In [ ]:
import torch.nn as nn

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            # 256 -> 128
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(0.2, inplace=True),
            # 128 -> 64
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            # 64 -> 32
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            # 32 -> 16
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            # 16 -> 15
            nn.Conv2d(512, 512, kernel_size=4, stride=1, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            # 15 -> 14
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)


# **Style loss**

In [ ]:
import torch
import torch.nn as nn
from torchvision.models import vgg19, VGG19_Weights
from torchvision import transforms

class StyleLoss(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.vgg = vgg19(weights=VGG19_Weights.DEFAULT).features.to(device).eval()
        self.selected_layers = ['1', '6', '11', '20', '29']
        self.weights = [0.244, 0.061, 0.015, 0.004, 0.004]

        for param in self.vgg.parameters():
            param.requires_grad = False

        self.transform = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                              std=[0.229, 0.224, 0.225])

    def gram_matrix(self, feature):
        (b, c, h, w) = feature.size()
        f = feature.view(b, c, h * w)
        G = torch.bmm(f, f.transpose(1, 2)) / (c * h * w)
        return G

    def forward(self, input, target):
        with torch.no_grad():
            input = self.transform(input)
            target = self.transform(target)

            style_loss = 0.0
            x = input
            y = target
            for i, layer in enumerate(self.vgg):
                x = layer(x)
                y = layer(y)
                if str(i) in self.selected_layers:
                    gm_x = self.gram_matrix(x)
                    gm_y = self.gram_matrix(y)
                    weight = self.weights[self.selected_layers.index(str(i))]
                    style_loss += weight * nn.functional.l1_loss(gm_x, gm_y)
        return style_loss


# **Perceptual loss**

In [ ]:
import torch.nn.functional as F
import torch
import torch.nn as nn
from torchvision import models

class PerceptualLoss(nn.Module):
    def __init__(self, device, layers=['relu3_3', 'relu4_3']):
        super().__init__()
        weights = models.VGG19_Weights.DEFAULT
        vgg = models.vgg19(weights=weights).features.to(device).eval()
        self.blocks = nn.ModuleList()
        self.layer_names = layers
        self.layer_indices = {'relu1_1': 0, 'relu1_2': 2, 'relu2_1': 5,
                              'relu2_2': 7, 'relu3_1': 10, 'relu3_2': 12,
                              'relu3_3': 14, 'relu3_4': 16, 'relu4_1': 19,
                              'relu4_2': 21, 'relu4_3': 23}

        prev_index = 0
        for name in layers:
            index = self.layer_indices[name]
            block = nn.Sequential(*list(vgg.children())[prev_index:index+1])
            for param in block.parameters():
                param.requires_grad = False
            self.blocks.append(block)
            prev_index = index + 1

        self.mean = torch.tensor([0.485, 0.456, 0.406]).to(device).view(1, 3, 1, 1)
        self.std = torch.tensor([0.229, 0.224, 0.225]).to(device).view(1, 3, 1, 1)

    def forward(self, input, target):
        with torch.no_grad():  # disabilita il tracking dei gradienti dentro questo blocco
            input = (input - self.mean) / self.std
            target = (target - self.mean) / self.std
            for block in self.blocks:
                input = block(input)
                target = block(target)
        # ora input e target sono i tensori delle feature estratte da VGG, senza grafo
        # calcoliamo la loss normalmente (qui serve il grafo solo se vuoi retropropagare su input)
        loss = F.l1_loss(input, target)
        return loss



# **GAN**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class GANTrainer:
    def __init__(self, generator, discriminator, device='cuda'):
        self.device = device

        # Usa i modelli passati come argomenti
        self.generator = generator.to(device)
        self.discriminator = discriminator.to(device)

        # Ottimizzatori
        self.optim_G = optim.Adam(self.generator.parameters(), lr=2e-4, betas=(0.5, 0.999))
        self.optim_D = optim.Adam(self.discriminator.parameters(), lr=2e-4, betas=(0.5, 0.999))

        # Loss functions
        self.adv_loss = nn.BCELoss()
        self.l1_loss = nn.L1Loss()

        # StyleLoss
        self.style_loss = StyleLoss(device).to(device)

        # PerceptualLoss
        self.perceptual_loss = PerceptualLoss(device=device)

        # Hyperparametri
        self.lambda_l1 = 50
        self.lambda_style = 10
        self.lambda_perceptual = 1

    def train_step(self, real_128, real_256, coords, return_generated=False):
        batch_size = real_128.size(0)
        real_128 = real_128.to(self.device)
        real_256 = real_256.to(self.device)

        valid = torch.ones((batch_size, 1, 14, 14), device=self.device)
        fake = torch.zeros((batch_size, 1, 14, 14), device=self.device)

        # Genera fake_256 subito
        fake_256 = self.generator(real_128)

        # === Patch localizzata da real_256 e fake_256 ===
        localized_real = torch.zeros((batch_size, 3, 128, 128), device=self.device)
        localized_fake = torch.zeros((batch_size, 3, 128, 128), device=self.device)

        sx_list, sy_list = coords
        for i in range(batch_size):
            sx = sx_list[i].item()
            sy = sy_list[i].item()
            localized_real[i] = real_256[i, :, sy:sy+128, sx:sx+128]
            localized_fake[i] = fake_256[i, :, sy:sy+128, sx:sx+128]

        # === Train Discriminator ===
        self.optim_D.zero_grad()
        pred_real = self.discriminator(real_256)
        pred_fake = self.discriminator(fake_256.detach())

        threshold = 0.5
        with torch.no_grad():
            real_acc = (pred_real > threshold).float().mean().item()
            fake_acc = (pred_fake < threshold).float().mean().item()

        loss_D = self.adv_loss(pred_real, valid) + self.adv_loss(pred_fake, fake)
        loss_D.backward()
        self.optim_D.step()

        # === Train Generator ===
        self.optim_G.zero_grad()
        pred_fake = self.discriminator(fake_256)
        loss_G_adv = self.adv_loss(pred_fake, valid)
        loss_L1 = self.l1_loss(localized_fake, localized_real)
        loss_style = self.style_loss(fake_256, real_256)
        loss_perceptual = self.perceptual_loss(fake_256, real_256)
        loss_G = loss_G_adv + self.lambda_l1 * loss_L1 + self.lambda_style * loss_style + self.lambda_perceptual * loss_perceptual
        loss_G.backward()
        self.optim_G.step()

        training_info = {
            'loss_D': loss_D.item(),
            'loss_G': loss_G.item(),
            'loss_G_adv': loss_G_adv.item(),
            'loss_L1': loss_L1.item(),
            'loss_style': loss_style.item(),
            'loss_perceptual': loss_perceptual.item(),
            'acc_real': real_acc * 100,
            'acc_fake': fake_acc * 100
        }

        if return_generated:
            return training_info, fake_256
        else:
            return training_info

# **Data loader**

In [ ]:
from torch.utils.data import Dataset
from PIL import Image
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import random
import os

class TextureFolderDataset(Dataset):
    def __init__(self, folder_path, num_samples=10000, transform=None, use_augmentation=False):
        """
        folder_path: Cartella contenente le immagini da cui estrarre i blocchi.
        num_samples: Numero di campioni totali da generare.
        transform: Trasformazioni da applicare ai blocchi (es. ToTensor, Normalize).
        use_augmentation: Se True, applica data augmentation sui blocchi.
        """
        self.image_paths = [os.path.join(folder_path, f) for f in os.listdir(folder_path)
                            if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        if not self.image_paths:
            raise ValueError(f"Nessuna immagine trovata in {folder_path}")

        self.images = [Image.open(p).convert('RGB') for p in self.image_paths]
        self.num_samples = num_samples
        self.k = 128
        self.use_augmentation = use_augmentation

        if transform:
            self.transform = transform
        else:
            # ToTensor sarà applicato comunque alla fine
            self.transform = None

    def __len__(self):
        return self.num_samples

    def apply_augmentation(self, image):
        # Data augmentation sincronizzata
        if self.use_augmentation:
            if random.random() < 0.5:
                image = TF.hflip(image)
            if random.random() < 0.5:
                image = TF.vflip(image)
            image = TF.adjust_brightness(image, 1 + (random.uniform(-0.2, 0.2)))
            image = TF.adjust_contrast(image, 1 + (random.uniform(-0.2, 0.2)))
            image = TF.adjust_saturation(image, 1 + (random.uniform(-0.2, 0.2)))
            image = TF.adjust_hue(image, random.uniform(-0.1, 0.1))
            angle = random.uniform(-10, 10)
            image = TF.rotate(image, angle)
        return image

    def __getitem__(self, idx):
        image = random.choice(self.images)
        width, height = image.size

        if width < 256 or height < 256:
            raise ValueError(f"L'immagine è troppo piccola: {width}x{height} - {image}")

        x = random.randint(0, width - 256)
        y = random.randint(0, height - 256)
        target_block = image.crop((x, y, x + 256, y + 256))

        # Applica le stesse trasformazioni al target prima di ricavare il source
        target_block = self.apply_augmentation(target_block)

        s_x = random.randint(0, 256 - 128)
        s_y = random.randint(0, 256 - 128)

        source_block = target_block.crop((s_x, s_y, s_x + 128, s_y + 128))

        # Applica ToTensor (e trasformazione finale opzionale se fornita)
        if self.transform:
            source_tensor = self.transform(source_block)
            target_tensor = self.transform(target_block)
        else:
            to_tensor = T.ToTensor()
            source_tensor = to_tensor(source_block)
            target_tensor = to_tensor(target_block)

        return source_tensor, target_tensor, (s_x, s_y)


# **Train**

In [ ]:
from datetime import datetime
import os
import torch
import matplotlib.pyplot as plt
import time
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm import tqdm

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    os.makedirs("/content/drive/MyDrive/7ImmAugmentation/results/weights", exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    checkpoint_path = f"/content/drive/MyDrive/7ImmAugmentation/results/weights/weights_{timestamp}.pt"

    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
    ])

    folder_path = '/content/drive/MyDrive/7ImmAugmentation/data/input_images/'
    num_files = sum([len(files) for _, _, files in os.walk(folder_path)])
    print(f"Numero totale di file nella cartella: {num_files}")

    dataset = TextureFolderDataset(folder_path='/content/drive/MyDrive/7ImmAugmentation/data/input_images/', num_samples=500000, transform=transform, use_augmentation=True)
    dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

    generator = Generator()
    discriminator = Discriminator()


    metrics_history = []
    start_step, losses, accuracies, metrics_history = load_weights(generator, discriminator)

    if start_step == 0:
        print("Inizializzazione manuale dei pesi (weights_init_normal)")
        generator.apply(weights_init_normal)
        discriminator.apply(weights_init_normal)

    trainer = GANTrainer(generator, discriminator, device)

    num_steps = 40000
    initial_lr = 2e-4

    losses_D_list = losses.get('losses_D_list', [])
    losses_G_list = losses.get('losses_G_list', [])
    losses_G_adv_list = losses.get('losses_G_adv_list', [])
    losses_L1_list = losses.get('losses_L1_list', [])
    losses_style_list = losses.get('losses_style_list', [])
    losses_perceptual_list = losses.get('losses_perceptual_list', [])
    acc_real_list = accuracies.get('acc_real_list', [])
    acc_fake_list = accuracies.get('acc_fake_list', [])

    print(f"\n-- Inizio train --\n")
    data_iter = iter(dataloader)
    for step in range(start_step, num_steps):
        start = time.time()

        try:
            s128, t256, coords = next(data_iter)
        except StopIteration:
            data_iter = iter(dataloader)
            s128, t256, coords = next(data_iter)

        # Linear decay learning rate
        if step > 50000:
            lr = initial_lr * (1 - (step - 50000) / 50000)
            for g in trainer.optim_G.param_groups: g['lr'] = lr
            for d in trainer.optim_D.param_groups: d['lr'] = lr

        if step % 100 == 0:

            losses, generated = trainer.train_step(s128, t256, coords, return_generated=True)
            end = time.time()

            visualize_sample(s128, t256, generated, step=step)
            visualize_losses(losses_D_list, losses_G_list, timestamp, find_well_trained_step(losses_G_list, acc_real_list, acc_fake_list))
            visualize_generator_loss_components(losses_G_adv_list, losses_L1_list, losses_style_list, losses_perceptual_list, timestamp)
            visualize_discriminator_accuracy(acc_real_list, acc_fake_list, timestamp)

            # Valutazione Intraset metrics -------------------------------------
            metrics = evaluate_intraset(t256, generated, device=device)
            if step == 0:
                metrics_history = []
            metrics_history.append({
                'step': step,
                'psnr': metrics['psnr'],
                'ssim': metrics['ssim'],
                'lpips': metrics['lpips'],
                'fid': metrics['fid']
            })

            # Salva plot delle metriche ogni 1000 step
            if step % 100 == 0 and len(metrics_history) > 1:
                save_intraset_metrics_plot(metrics_history, num_files)

            # ------------------------------------------------------------------

            torch.save({
                'step': step,
                'generator_state_dict': generator.state_dict(),
                'discriminator_state_dict': discriminator.state_dict(),
                'optimizer_G_state_dict': trainer.optim_G.state_dict(),
                'optimizer_D_state_dict': trainer.optim_D.state_dict(),
                'losses_D_list': losses_D_list,
                'losses_G_list': losses_G_list,
                'losses_G_adv_list': losses_G_adv_list,
                'losses_L1_list': losses_L1_list,
                'losses_style_list': losses_style_list,
                'losses_perceptual_list': losses_perceptual_list,
                'acc_real_list': acc_real_list,
                'acc_fake_list': acc_fake_list,
                'metrics_history': metrics_history
            }, checkpoint_path)

        else:
            losses = trainer.train_step(s128, t256, coords)
            end = time.time()

        # Visualizza informazioni
        cpu_tensors, gpu_tensors = count_tensors()
        print(f"[{step}/{num_steps}] D: {losses['loss_D']:.4f} G: {losses['loss_G']:.4f} "
              f"(Adv: {losses['loss_G_adv']:.4f}, L1: {losses['loss_L1']:.4f}, Style: {losses['loss_style']:.4f}, Percep: {losses['loss_perceptual']:.4f}) "
              f"- {end - start:.3f}s | CPU tensors: {cpu_tensors}, GPU tensors: {gpu_tensors}")

        losses_D_list.append(losses['loss_D'])
        losses_G_list.append(losses['loss_G'])
        losses_G_adv_list.append(losses['loss_G_adv'])
        losses_L1_list.append(losses['loss_L1'])
        losses_style_list.append(losses['loss_style'])
        losses_perceptual_list.append(losses['loss_perceptual'])
        acc_real_list.append(losses['acc_real'])
        acc_fake_list.append(losses['acc_fake'])

    torch.save({
        'step': num_steps,
        'generator_state_dict': generator.state_dict(),
        'discriminator_state_dict': discriminator.state_dict(),
        'optimizer_G_state_dict': trainer.optim_G.state_dict(),
        'optimizer_D_state_dict': trainer.optim_D.state_dict(),
        'losses_D_list': losses_D_list,
        'losses_G_list': losses_G_list,
        'losses_G_adv_list': losses_G_adv_list,
        'losses_L1_list': losses_L1_list,
        'losses_style_list': losses_style_list,
        'losses_perceptual_list': losses_perceptual_list,
        'acc_real_list': acc_real_list,
        'acc_fake_list': acc_fake_list,
        'metrics_history': metrics_history
    }, checkpoint_path)


if __name__ == "__main__":
    main()

Using device: cuda
Numero totale di file nella cartella: 7
Nessun file di pesi trovato. Inizializzazione dei pesi.
Inizializzazione manuale dei pesi (weights_init_normal)


Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth
100%|██████████| 548M/548M [00:07<00:00, 74.6MB/s]



-- Inizio train --

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100%|██████████| 528M/528M [00:07<00:00, 71.4MB/s]


Loading model from: /usr/local/lib/python3.11/dist-packages/lpips/weights/v0.1/vgg.pth


/usr/local/lib/python3.11/dist-packages/torch/__init__.py:1113: FutureWarning: `torch.distributed.reduce_op` is deprecated, please use `torch.distributed.ReduceOp` instead
  return isinstance(obj, torch.Tensor)


[0/40000] D: 1.6952 G: 46.9080 (Adv: 0.9185, L1: 0.6455, Style: 0.0105, Percep: 13.6107) - 1.528s | CPU tensors: 103, GPU tensors: 534
[1/40000] D: 3.6993 G: 36.2509 (Adv: 1.0223, L1: 0.5041, Style: 0.0085, Percep: 9.9365) - 0.627s | CPU tensors: 103, GPU tensors: 534
[2/40000] D: 2.2554 G: 31.2413 (Adv: 1.0861, L1: 0.3956, Style: 0.0058, Percep: 10.3192) - 0.624s | CPU tensors: 103, GPU tensors: 534
[3/40000] D: 1.7970 G: 29.5202 (Adv: 1.2309, L1: 0.3205, Style: 0.0043, Percep: 12.2236) - 0.626s | CPU tensors: 103, GPU tensors: 534
[4/40000] D: 1.7163 G: 26.4066 (Adv: 1.0548, L1: 0.3055, Style: 0.0031, Percep: 10.0453) - 0.628s | CPU tensors: 103, GPU tensors: 534
[5/40000] D: 1.4915 G: 25.0836 (Adv: 1.2297, L1: 0.2463, Style: 0.0038, Percep: 11.4997) - 0.626s | CPU tensors: 103, GPU tensors: 534
[6/40000] D: 1.5230 G: 23.3280 (Adv: 1.1489, L1: 0.2520, Style: 0.0027, Percep: 9.5531) - 0.627s | CPU tensors: 103, GPU tensors: 534
[7/40000] D: 1.4967 G: 23.4600 (Adv: 1.5668, L1: 0.2209, 